# Análise do Mercado Automotivo Brasileiro
## Emplacamentos, rankings e avanço dos veículos eletrificados

**Período:** julho de 2025 a junho de 2026  
**Fonte:** informativos mensais da FENABRAVE

### Objetivo

Analisar a evolução dos emplacamentos de automóveis no Brasil, identificando:

- principais marcas e modelos presentes no Top 50 mensal;
- mudanças de posição e consistência no ranking;
- modelos que ganharam ou perderam força;
- evolução dos veículos híbridos e 100% elétricos;
- crescimento da participação dos eletrificados no mercado de automóveis.

### Escopo e limitações

A base de modelos contém os **50 automóveis mais emplacados em cada mês**.
Portanto, somas e participações calculadas a partir dessa base descrevem a
**amostra Top 50**, e não o mercado completo por marca.

Para a análise de eletrificação, a participação foi calculada em relação ao
**total de automóveis emplacados** informado nos relatórios da FENABRAVE.


## 1. Bibliotecas

Nesta etapa são importadas as bibliotecas usadas para manipulação dos dados,
cálculos e visualizações.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


## 2. Carregamento dos dados

Foram utilizadas duas bases construídas a partir dos informativos mensais da
FENABRAVE: uma com os **50 modelos de automóveis mais emplacados por mês** e
outra com os totais mensais de **híbridos e elétricos**.


In [ ]:
# Carregamento das bases
# Estrutura esperada no repositório:
# projeto/
# ├── AnaliseMercadoAutomotivo_GitHub.ipynb
# └── dados/
#     ├── automoveis_fenabrave_jul2025_jun2026.csv
#     └── eletrificados_fenabrave_jul2025_jun2026.csv

import os

if not os.path.exists("dados"):
    !git clone -q https://github.com/jefferson1337br/analise-mercado-automotivo.git
    os.chdir("analise-mercado-automotivo")

print("Diretório atual:", os.getcwd())

dados_automoveis = pd.read_csv(
    "dados/automoveis_fenabrave_jul2025_jun2026.csv"
)

dados_eletrificados = pd.read_csv(
    "dados/eletrificados_fenabrave_jul2025_jun2026.csv"
)


Diretório atual: /content/analise-mercado-automotivo


## 3. Exploração e validação inicial

Antes da análise, são verificadas a cobertura temporal, estatísticas descritivas,
tipos de dados e possíveis inconsistências.


In [ ]:
dados_automoveis['periodo'].unique()
dados_automoveis.groupby('periodo').size()

In [ ]:
dados_automoveis.describe()

In [ ]:
dados_automoveis['emplacamentos'].describe()

## 4. Tratamento dos dados

É criada uma cópia das bases originais, os textos de marca e modelo são
padronizados e o período é convertido para uma coluna de data.


In [ ]:
dados_automoveis_tratados=dados_automoveis.copy()
dados_eletrificados_tratados=dados_eletrificados.copy()

In [ ]:
dados_automoveis_tratados['marca'] = (
    dados_automoveis_tratados['marca'].str.strip().str.upper()
)

dados_automoveis_tratados['modelo'] = (
    dados_automoveis_tratados['modelo'].str.strip().str.upper()
)


In [ ]:
dados_automoveis_tratados['data']=pd.to_datetime(dados_automoveis_tratados['periodo']+'-01')
dados_eletrificados_tratados['data']=pd.to_datetime(dados_eletrificados_tratados['periodo']+'-01')


In [ ]:
dados_automoveis_tratados[['periodo','data']].head()

In [ ]:
dados_automoveis_tratados.info()

In [ ]:
dados_automoveis_tratados['emplacamentos'].min()

In [ ]:
dados_automoveis_tratados[dados_automoveis_tratados['emplacamentos']<=0]

In [ ]:
(
    dados_automoveis_tratados['posicao'].min(),
    dados_automoveis_tratados['posicao'].max()
)


In [ ]:
dados_automoveis_tratados.groupby('periodo')['posicao'].nunique()

## 5. Evolução mensal dos modelos presentes no Top 50

> **Importante:** a soma abaixo representa os emplacamentos dos modelos que
> aparecem no Top 50 mensal. Ela não corresponde ao total oficial de automóveis
> emplacados no Brasil.


In [ ]:
emplacamento_mes=(
    dados_automoveis_tratados
    .groupby
    (['data','periodo'])['emplacamentos']
    .sum()
    .reset_index()
    .sort_values('data')

)


In [ ]:
emplacamento_mes.loc[
    emplacamento_mes['emplacamentos'].idxmax()
]

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(
    emplacamento_mes['data']
    ,emplacamento_mes['emplacamentos']
    ,marker='o')
plt.title('Evolução mensal dos emplacamentos- top 50 automoveis')
plt.xlabel('Mes')
plt.ylabel('Emplacamentos')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
emplacamento_mes['variacao_pct']=(
    emplacamento_mes['emplacamentos']
    .pct_change()*100
)
emplacamento_mes

In [ ]:
emplacamento_mes['variacao_pct']=emplacamento_mes['variacao_pct'].round(2)


In [ ]:
emplacamento_mes[
    ['periodo','emplacamentos','variacao_pct']
    ]

In [ ]:
primeiro_valor=emplacamento_mes.iloc[0]['emplacamentos']
ultimo_valor=emplacamento_mes.iloc[-1]['emplacamentos']

crescimento_periodo=((ultimo_valor-primeiro_valor)
/primeiro_valor)*100

print(f"Variação entre o primeiro e o ultimo mes é de {crescimento_periodo:.2f}%")

## 6. Análise por marca

Nesta seção são avaliados volume acumulado, quantidade de modelos presentes no
ranking e evolução mensal das principais marcas.

> A participação calculada nesta seção é a participação **dentro da amostra
> Top 50**, e não o market share oficial da marca no mercado total.


In [ ]:
ranking_marca=(
    dados_automoveis_tratados
    .groupby('marca')['emplacamentos']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
ranking_marca

In [ ]:
top10_marca=ranking_marca.head(10)
top10_marca

In [ ]:
plt.figure(figsize=(15,5))
plt.bar(
    top10_marca['marca']
    ,top10_marca['emplacamentos']
    ,color='blue'
)
plt.title('Top 10 Marcas com mais emplacamentos')
plt.xlabel('Marca')
plt.ylabel('Emplacamentos')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.show()


In [ ]:
total_emplacamentos=dados_automoveis_tratados['emplacamentos'].sum()
total_emplacamentos

In [ ]:
ranking_marca['participacao_pct']=(
    ranking_marca['emplacamentos']/total_emplacamentos*100
)
ranking_marca['participacao_pct']=(ranking_marca['participacao_pct'].round(2))
ranking_marca.head(10)

In [ ]:
modelos_por_marca=(
    dados_automoveis_tratados
    .groupby('marca')['modelo']
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="quantidade_modelos")
)
modelos_por_marca
#

In [ ]:
ranking_marca_completo=ranking_marca.merge(
    modelos_por_marca,on='marca'

)
ranking_marca_completo.head(10)

In [ ]:
marca_mes=(
  dados_automoveis_tratados
  .groupby(['data','periodo','marca'])['emplacamentos']
  .sum()
  .reset_index()
  .sort_values('data')
)
marca_mes.head(20)

In [ ]:
top5_marcas=(ranking_marca.head(5)['marca'].tolist())
top5_marcas

In [ ]:
evolucao_top5=marca_mes[marca_mes['marca'].isin(top5_marcas)]
evolucao_top5.head()

In [ ]:
plt.figure(figsize=(15,5))
for marca in top5_marcas:
  dados_marca=evolucao_top5[evolucao_top5['marca']==marca]
  plt.plot(
      dados_marca['data']
      ,dados_marca['emplacamentos']
      ,marker='o'
      ,label=marca
  )
plt.title('Evolução mensal dos emplacamentos- top 5 marcas')
plt.xlabel('Mes')
plt.ylabel('Emplacamentos')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.legend
plt.show()

In [ ]:
total_mensal=(
    dados_automoveis_tratados
    .groupby('data')['emplacamentos']
    .sum()
    .reset_index(name="total_mes")
)
total_mensal


In [ ]:
marca_mes=marca_mes.merge(total_mensal,on='data')

In [ ]:
marca_mes['participacao_top50_pct']=(
    marca_mes['emplacamentos']/marca_mes['total_mes']*100
)
marca_mes['participacao_top50_pct']=(marca_mes['participacao_top50_pct'].round(2))

marca_mes.head(10)


In [ ]:
share_top5=marca_mes[marca_mes['marca'].isin(top5_marcas)]


In [ ]:
plt.figure(figsize=(15,5))
for marca in top5_marcas:
  dados_marca=share_top5[share_top5['marca']==marca]
  plt.plot(
      dados_marca['data']
      ,dados_marca['participacao_top50_pct']
      ,marker='o'
      ,label=marca
  )
plt.title('Evolução da Participação das 5 Principais Marcas no Top 50')
plt.xlabel('Mes')
plt.ylabel('Participação (%)')
plt.legend()
plt.grid(alpha=0.3)
plt.show


In [ ]:
share_inicio = (
    marca_mes[
        marca_mes['periodo'] == '2025-07'
    ][['marca', 'participacao_top50_pct']]
    .rename(
        columns={
            'participacao_top50_pct': 'share_inicio'
        }
    )
)


In [ ]:
share_final = (
    marca_mes[
        marca_mes['periodo'] == '2026-06'
    ][['marca', 'participacao_top50_pct']]
    .rename(
        columns={
            'participacao_top50_pct': 'share_final'
        }
    )
)


In [ ]:
variacao_share = share_inicio.merge(
    share_final,
    on='marca',
    how='outer'
)


In [ ]:
variacao_share['variacao_pontos_percentuais'] = (
    variacao_share['share_final']
    - variacao_share['share_inicio']
)

variacao_share.sort_values(
    'variacao_pontos_percentuais',
    ascending=False
)


In [ ]:
maiores_altas = (
    variacao_share
    .dropna()
    .sort_values(
        'variacao_pontos_percentuais',
        ascending=False
    )
    .head(5)
)

maiores_altas


In [ ]:
maiores_quedas = (
    variacao_share
    .dropna()
    .sort_values(
        'variacao_pontos_percentuais'
    )
    .head(5)
)

maiores_quedas


## 7. Análise por modelo

A análise por modelo busca identificar liderança, consistência, posição média,
crescimento e mudanças na composição do Top 50.


In [ ]:
ranking_modelos = (
    dados_automoveis_tratados
    .groupby(['marca', 'modelo'])['emplacamentos']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

ranking_modelos.head(10)

In [ ]:
top10_modelos = ranking_modelos.head(10)

top10_modelos

In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    top10_modelos['modelo'],
    top10_modelos['emplacamentos']
)

plt.title('Top 10 Modelos por Emplacamentos no Período')
plt.xlabel('Modelo')
plt.ylabel('Emplacamentos')

plt.xticks(rotation=45)

plt.show()

In [ ]:
media_modelos = (
    dados_automoveis_tratados
    .groupby(['marca', 'modelo'])['emplacamentos']
    .mean()
    .sort_values(ascending=False)
    .reset_index(name='media_mensal')
)

media_modelos.head(10)

In [ ]:
lideres_mensais = dados_automoveis_tratados[
    dados_automoveis_tratados['posicao'] == 1
]

lideres_mensais[
    ['periodo', 'marca', 'modelo', 'emplacamentos']
]

In [ ]:
vezes_primeiro = (
    lideres_mensais
    .groupby(['marca', 'modelo'])
    .size()
    .sort_values(ascending=False)
    .reset_index(name='vezes_em_primeiro')
)

vezes_primeiro

In [ ]:
top5_mensal = dados_automoveis_tratados[
    dados_automoveis_tratados['posicao'] <= 5
]

presenca_top5 = (
    top5_mensal
    .groupby(['marca', 'modelo'])
    .size()
    .sort_values(ascending=False)
    .reset_index(name='meses_no_top5')
)

presenca_top5.head(10)

In [ ]:
top10_mensal = dados_automoveis_tratados[
    dados_automoveis_tratados['posicao'] <= 10
]

presenca_top10 = (
    top10_mensal
    .groupby(['marca', 'modelo'])
    .size()
    .sort_values(ascending=False)
    .reset_index(name='meses_no_top10')
)

presenca_top10.head(10)

In [ ]:
posicao_media = (
    dados_automoveis_tratados
    .groupby(['marca', 'modelo'])['posicao']
    .mean()
    .sort_values()
    .reset_index(name='posicao_media')
)

posicao_media.head(10)

In [ ]:
ranking_modelos_completo = (
    ranking_modelos
    .merge(
        media_modelos,
        on=['marca', 'modelo']
    )
    .merge(
        posicao_media,
        on=['marca', 'modelo']
    )
)

ranking_modelos_completo.head(10)

In [ ]:
ranking_modelos_completo = (
    ranking_modelos_completo
    .merge(
        presenca_top10,
        on=['marca', 'modelo'],
        how='left'
    )
)

ranking_modelos_completo['meses_no_top10'] = (
    ranking_modelos_completo['meses_no_top10']
    .fillna(0)
)

ranking_modelos_completo.head(10)

In [ ]:
polo = dados_automoveis_tratados[
    (dados_automoveis_tratados['marca'] == 'VW') &
    (dados_automoveis_tratados['modelo'] == 'POLO')
]

polo[
    ['periodo', 'posicao', 'emplacamentos']
]

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    polo['data'],
    polo['emplacamentos'],
    marker='o'
)

plt.title('Evolução Mensal dos Emplacamentos - VW Polo')
plt.xlabel('Mês')
plt.ylabel('Emplacamentos')

plt.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    polo['data'],
    polo['posicao'],
    marker='o'
)

plt.gca().invert_yaxis()

plt.title('Evolução da Posição no Ranking - VW Polo')
plt.xlabel('Mês')
plt.ylabel('Posição')

plt.grid(alpha=0.3)

plt.show()

In [ ]:
modelos_comparacao = [
    'POLO',
    'ARGO',
    'T CROSS',
    'COROLLA CROSS'
]

In [ ]:
dados_comparacao = dados_automoveis_tratados[
    dados_automoveis_tratados['modelo'].isin(modelos_comparacao)
]

In [ ]:
plt.figure(figsize=(14, 7))

for modelo in modelos_comparacao:

    dados_modelo = dados_comparacao[
        dados_comparacao['modelo'] == modelo
    ]

    plt.plot(
        dados_modelo['data'],
        dados_modelo['emplacamentos'],
        marker='o',
        label=modelo
    )

plt.title('Evolução Mensal de Modelos Selecionados')
plt.xlabel('Mês')
plt.ylabel('Emplacamentos')

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
inicio_modelos = (
    dados_automoveis_tratados[
        dados_automoveis_tratados['periodo'] == '2025-07'
    ][['marca', 'modelo', 'emplacamentos']]
    .rename(
        columns={
            'emplacamentos': 'emplacamentos_inicio'
        }
    )
)

In [ ]:
fim_modelos = (
    dados_automoveis_tratados[
        dados_automoveis_tratados['periodo'] == '2026-06'
    ][['marca', 'modelo', 'emplacamentos']]
    .rename(
        columns={
            'emplacamentos': 'emplacamentos_fim'
        }
    )
)

In [ ]:
crescimento_modelos = inicio_modelos.merge(
    fim_modelos,
    on=['marca', 'modelo'],
    how='inner'
)

In [ ]:
crescimento_modelos['variacao_absoluta'] = (
    crescimento_modelos['emplacamentos_fim']
    - crescimento_modelos['emplacamentos_inicio']
)

In [ ]:
crescimento_modelos['variacao_pct'] = (
    (
        crescimento_modelos['emplacamentos_fim']
        - crescimento_modelos['emplacamentos_inicio']
    )
    / crescimento_modelos['emplacamentos_inicio']
) * 100

In [ ]:
crescimento_modelos['variacao_pct'] = (
    crescimento_modelos['variacao_pct']
    .round(2)
)

In [ ]:
crescimento_modelos.sort_values(
    'variacao_pct',
    ascending=False
).head(10)

In [ ]:
crescimento_modelos.sort_values(
    'variacao_pct'
).head(10)

In [ ]:
modelos_inicio = set(
    dados_automoveis_tratados[
        dados_automoveis_tratados['periodo'] == '2025-07'
    ]['modelo']
)

In [ ]:
modelos_fim = set(
    dados_automoveis_tratados[
        dados_automoveis_tratados['periodo'] == '2026-06'
    ]['modelo']
)

In [ ]:
novos_modelos = modelos_fim - modelos_inicio

novos_modelos

In [ ]:
modelos_que_sairam = modelos_inicio - modelos_fim

modelos_que_sairam

## 8. Mercado de veículos eletrificados

Nesta etapa é utilizada a base agregada de híbridos e elétricos para avaliar a
evolução do volume e da participação dos eletrificados no mercado de
automóveis.


In [ ]:
dados_eletrificados_tratados.head()

In [ ]:
dados_eletrificados_tratados.info()

In [ ]:
dados_eletrificados_tratados[
    [
        'periodo',
        'total_automoveis',
        'hibridos',
        'eletricos',
        'total_eletrificados'
    ]
]

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    dados_eletrificados_tratados['data'],
   dados_eletrificados_tratados['total_eletrificados'],
    marker='o'
)

plt.title('Evolução Mensal dos Veículos Eletrificados')
plt.xlabel('Mês')
plt.ylabel('Emplacamentos')

plt.grid(alpha=0.3)

plt.show()

In [ ]:
inicio = dados_eletrificados_tratados.iloc[0]['total_eletrificados']
fim = dados_eletrificados_tratados.iloc[-1]['total_eletrificados']

crescimento_eletrificados = (
    (fim - inicio) / inicio
) * 100

print(
    f'Crescimento dos eletrificados no período: '
    f'{crescimento_eletrificados:.2f}%'
)

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    dados_eletrificados_tratados['data'],
    dados_eletrificados_tratados['hibridos'],
    marker='o',
    label='Híbridos'
)

plt.plot(
    dados_eletrificados_tratados['data'],
    dados_eletrificados_tratados['eletricos'],
    marker='o',
    label='Elétricos'
)

plt.title('Evolução Mensal: Híbridos x Elétricos')
plt.xlabel('Mês')
plt.ylabel('Emplacamentos')

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
crescimento_hibridos = (
    (
        dados_eletrificados_tratados.iloc[-1]['hibridos']
        - dados_eletrificados_tratados.iloc[0]['hibridos']
    )
    / dados_eletrificados_tratados.iloc[0]['hibridos']
) * 100

crescimento_hibridos

In [ ]:
crescimento_eletricos = (
    (
        dados_eletrificados_tratados.iloc[-1]['eletricos']
        - dados_eletrificados_tratados.iloc[0]['eletricos']
    )
    / dados_eletrificados_tratados.iloc[0]['eletricos']
) * 100

crescimento_eletricos

In [ ]:
print(f'Crescimento dos híbridos: {crescimento_hibridos:.2f}%')
print(f'Crescimento dos elétricos: {crescimento_eletricos:.2f}%')

In [ ]:
dados_eletrificados_tratados[
    [
        'periodo',
        'participacao_eletrificados_pct'
    ]
]

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
   dados_eletrificados_tratados['data'],
    dados_eletrificados_tratados['participacao_eletrificados_pct'],
    marker='o'
)

plt.title('Participação dos Eletrificados no Mercado de Automóveis')
plt.xlabel('Mês')
plt.ylabel('Participação (%)')

plt.grid(alpha=0.3)

plt.show()

In [ ]:
share_inicio = (
    dados_eletrificados_tratados
    .iloc[0]['participacao_eletrificados_pct']
)

share_final = (
    dados_eletrificados_tratados
    .iloc[-1]['participacao_eletrificados_pct']
)

ganho_share = share_final - share_inicio

print(f'Participação inicial: {share_inicio:.2f}%')
print(f'Participação final: {share_final:.2f}%')
print(f'Ganho de participação: {ganho_share:.2f} p.p.')

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    dados_eletrificados_tratados['data'],
   dados_eletrificados_tratados['participacao_hibridos_pct'],
    marker='o',
    label='Híbridos'
)

plt.plot(
dados_eletrificados_tratados['data'],
    dados_eletrificados_tratados['participacao_eletricos_pct'],
    marker='o',
    label='Elétricos'
)

plt.title('Participação de Híbridos e Elétricos no Mercado')
plt.xlabel('Mês')
plt.ylabel('Participação (%)')

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
dados_eletrificados_tratados['crescimento_eletrificados_pct'] = (
    dados_eletrificados_tratados['total_eletrificados']
    .pct_change() * 100
)

dados_eletrificados_tratados['crescimento_eletricos_pct'] = (
    dados_eletrificados_tratados['eletricos']
    .pct_change() * 100
)

dados_eletrificados_tratados['crescimento_hibridos_pct'] = (
    dados_eletrificados_tratados['hibridos']
    .pct_change() * 100
)

In [ ]:
colunas_crescimento = [
    'crescimento_eletrificados_pct',
    'crescimento_eletricos_pct',
    'crescimento_hibridos_pct'
]

dados_eletrificados_tratados[colunas_crescimento] = (
   dados_eletrificados_tratados[colunas_crescimento]
    .round(2)
)

In [ ]:
dados_eletrificados_tratados[
    [
        'periodo',
        'total_eletrificados',
        'crescimento_eletrificados_pct'
    ]
]

In [ ]:
maior_share = dados_eletrificados_tratados.loc[
    dados_eletrificados_tratados[
        'participacao_eletrificados_pct'
    ].idxmax()
]

maior_share[
    [
        'periodo',
        'participacao_eletrificados_pct'
    ]
]

In [ ]:
menor_share = dados_eletrificados_tratados.loc[
    dados_eletrificados_tratados[
        'participacao_eletrificados_pct'
    ].idxmin()
]

menor_share[
    [
        'periodo',
        'participacao_eletrificados_pct'
    ]
]

In [ ]:
dados_eletrificados_tratados['share_hibridos_eletrificados'] = (
    dados_eletrificados_tratados['hibridos']
    / dados_eletrificados_tratados['total_eletrificados']
    * 100
)

dados_eletrificados_tratados['share_eletricos_eletrificados'] = (
    dados_eletrificados_tratados['eletricos']
    / dados_eletrificados_tratados['total_eletrificados']
    * 100
)

In [ ]:
dados_eletrificados_tratados[
    [
        'periodo',
        'share_hibridos_eletrificados',
        'share_eletricos_eletrificados'
    ]
]

In [ ]:
dados_eletrificados_tratados['indice_mercado'] = (
    dados_eletrificados_tratados['total_automoveis']
    / dados_eletrificados_tratados.iloc[0]['total_automoveis']
    * 100
)

dados_eletrificados_tratados['indice_eletrificados'] = (
    dados_eletrificados_tratados['total_eletrificados']
    / dados_eletrificados_tratados.iloc[0]['total_eletrificados']
    * 100
)

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    dados_eletrificados_tratados['data'],
    dados_eletrificados_tratados['indice_mercado'],
    marker='o',
    label='Mercado total'
)

plt.plot(
    dados_eletrificados_tratados['data'],
    dados_eletrificados_tratados['indice_eletrificados'],
    marker='o',
    label='Eletrificados'
)

plt.axhline(
    y=100,
    linestyle='--',
    alpha=0.5
)

plt.title('Crescimento Relativo: Mercado Total x Eletrificados')
plt.xlabel('Mês')
plt.ylabel('Índice (Jul/2025 = 100)')

plt.legend()
plt.grid(alpha=0.3)

plt.show()

## 9. Aceleração e desaceleração da eletrificação

Além do crescimento acumulado, analisamos a mudança mensal da participação para
identificar períodos de avanço e retração.


In [ ]:
evolucao_eletrificados = dados_eletrificados_tratados[
    [
        'periodo',
        'hibridos',
        'eletricos',
        'total_eletrificados',
        'participacao_eletrificados_pct'
    ]
].copy()

evolucao_eletrificados

In [ ]:
evolucao_eletrificados['variacao_share_pp'] = (
    evolucao_eletrificados[
        'participacao_eletrificados_pct'
    ].diff()
)

evolucao_eletrificados[
    [
        'periodo',
        'participacao_eletrificados_pct',
        'variacao_share_pp'
    ]
]

In [ ]:
plt.figure(figsize=(14, 7))

plt.plot(
    dados_eletrificados_tratados['data'],
    dados_eletrificados_tratados['participacao_eletrificados_pct'],
    marker='o',
    linewidth=2
)

plt.title(
    'Evolução da Participação dos Veículos Eletrificados'
)
plt.xlabel('Mês')
plt.ylabel('Participação no mercado (%)')

plt.grid(alpha=0.3)
plt.xticks(rotation=45)

plt.show()

In [ ]:
plt.figure(figsize=(14, 7))

plt.plot(
    dados_eletrificados_tratados['data'],
    dados_eletrificados_tratados['participacao_eletrificados_pct'],
    marker='o',
    linewidth=2
)

for x, y in zip(
    dados_eletrificados_tratados['data'],
    dados_eletrificados_tratados['participacao_eletrificados_pct']
):
    plt.text(
        x,
        y + 0.3,
        f'{y:.1f}%',
        ha='center',
        fontsize=9
    )

plt.title(
    'Evolução da Participação dos Veículos Eletrificados'
)
plt.xlabel('Mês')
plt.ylabel('Participação no mercado (%)')

plt.grid(alpha=0.3)
plt.xticks(rotation=45)

plt.show()

In [ ]:
maiores_avancos = (
    evolucao_eletrificados
    .sort_values(
        'variacao_share_pp',
        ascending=False
    )
    .head(3)
)

maiores_avancos[
    [
        'periodo',
        'participacao_eletrificados_pct',
        'variacao_share_pp'
    ]
]

In [ ]:
maiores_quedas = (
    evolucao_eletrificados
    .dropna()
    .sort_values(
        'variacao_share_pp'
    )
    .head(3)
)

maiores_quedas[
    [
        'periodo',
        'participacao_eletrificados_pct',
        'variacao_share_pp'
    ]
]

## 10. Principais conclusões

Entre julho de 2025 e junho de 2026, os veículos eletrificados apresentaram
forte expansão:

- **Eletrificados:** +114,60%
- **Híbridos:** +80,75%
- **100% elétricos:** +202,29%
- **Participação dos eletrificados:** de 13,73% para 25,17%
- **Ganho de participação:** +11,44 pontos percentuais

O avanço não foi linear. Houve retrações em alguns meses, mas a série terminou
no maior nível de participação observado. O maior ganho mensal ocorreu em
dezembro de 2025 (+4,46 p.p.), seguido por abril de 2026 (+3,88 p.p.) e janeiro
de 2026 (+3,09 p.p.).

Os resultados indicam que os eletrificados não apenas cresceram em volume:
eles também conquistaram uma parcela maior do mercado de automóveis no período,
com destaque para o ritmo de crescimento dos veículos 100% elétricos.

### Limitações

- A análise de marcas e modelos utiliza somente os modelos presentes no Top 50
  mensal.
- Um modelo fora do Top 50 em determinado mês não possui observação nessa base.
- Comparações de crescimento entre julho/2025 e junho/2026 consideram apenas
  modelos presentes nos dois meses quando utilizado `merge(..., how='inner')`.
- A análise de eletrificação é agregada e não permite identificar, nesta base,
  quais fabricantes ou modelos eletrificados explicam o crescimento.
